# **Proyecto Sprint 7: Análisis del mercado automotriz (EDA)**

## **Descripción :** 

Este notebook presenta el análisis exploratorio de datos (EDA) y la preparación del dataset para la aplicación web interactiva, llevando a cabo procesamientos de datawrangling y preprocesamiento de datos para visualizar las tendencias clave de precios, condiciones y tipos de unidades.

### Configuración de entorno e importación de librerías

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import os
import sys

### Configuración de visualización para compatibilidad con VS Code

In [3]:
if "vscode" in pio.renderers.default:
    pio.renderers.default = "notebook_connected"
else:
    pio.renderers.default = "vscode"

### Verificación de entorno


In [4]:
print(f"Cerebro activo en: {sys.executable}")
print(f"Directorio de trabajo: {os.getcwd()}")

Cerebro activo en: c:\Users\alxsc\anaconda3\envs\Tensorflow\python.exe
Directorio de trabajo: c:\Users\alxsc\Desktop\Personal\TripleTen - Sprint 7\notebooks


### Carga de datos

In [10]:
df_vehicles = pd.read_csv("../vehicles_us.csv")
df_vehicles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  object 
 3   condition     51525 non-null  object 
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  object 
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  object 
 8   type          51525 non-null  object 
 9   paint_color   42258 non-null  object 
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  object 
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), object(7)
memory usage: 5.1+ MB


### Data wrangling

Para asegurar un análisis preciso en la aplicación web, se realizaron las siguientes transformaciones: 1) Imputación de valores mediante la mediana por grupo o moda; 2) Conversión a enteros para mejorar el rendimiento; y 3) Clasificación de colores faltantes y normalización de tracción.

In [11]:
# Tracción (is_4wd): Si es nulo, es porque no tiene (0)
df_vehicles['is_4wd'] = df_vehicles['is_4wd'].fillna(0).astype(int)

# Año del modelo: Llenar nulos con la mediana por modelo
df_vehicles['model_year'] = df_vehicles['model_year'].fillna(
df_vehicles.groupby('model')['model_year'].transform('median')
)
df_vehicles.dropna(subset=['model_year'], inplace=True)
df_vehicles['model_year'] = df_vehicles['model_year'].astype(int)

# Cilindros: Llenar con la moda por modelo
df_vehicles['cylinders'] = df_vehicles['cylinders'].fillna(
df_vehicles.groupby('model')['cylinders'].transform(
lambda x: x.mode()[0] if not x.mode().empty else np.nan
)
)

# Odómetro (kilometraje): Llenar con la mediana según el año del modelo
df_vehicles['odometer'] = df_vehicles['odometer'].fillna(
df_vehicles.groupby('model_year')['odometer'].transform('median')
)
df_vehicles['odometer'] = df_vehicles['odometer'].fillna(df_vehicles['odometer'].median())

# Color y conversión final
df_vehicles['paint_color'] = df_vehicles['paint_color'].fillna('unknown')
df_vehicles['cylinders'] = df_vehicles['cylinders'].fillna(df_vehicles['cylinders'].median()).astype(int)
df_vehicles['odometer'] = df_vehicles['odometer'].astype(int)

df_vehicles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   price         51525 non-null  int64 
 1   model_year    51525 non-null  int64 
 2   model         51525 non-null  object
 3   condition     51525 non-null  object
 4   cylinders     51525 non-null  int64 
 5   fuel          51525 non-null  object
 6   odometer      51525 non-null  int64 
 7   transmission  51525 non-null  object
 8   type          51525 non-null  object
 9   paint_color   51525 non-null  object
 10  is_4wd        51525 non-null  int64 
 11  date_posted   51525 non-null  object
 12  days_listed   51525 non-null  int64 
dtypes: int64(6), object(7)
memory usage: 5.1+ MB


### Visualización de precios

In [13]:
fig1 = px.histogram(df_vehicles, x="price", nbins=50,
title="Distribución de precios de venta",
labels={'price': 'Precio ($)', 'count': 'Frecuencia'},
color_discrete_sequence=['indianred'])
fig1.show()

### Visualización de datos

In [15]:
# Histograma de distribución de precios de venta
fig1 = px.histogram(df_vehicles, x="price", nbins=50, 
                   title="Distribución de precios de venta",
                   labels={'price': 'Precio ($)'}, # Quitamos count de aquí
                   color_discrete_sequence=['indianred'])

# El truco maestro para el eje Y:
fig1.update_layout(
    xaxis_title="Precio ($)",
    yaxis_title="Frecuencia",
    bargap=0.05 # Un pequeño espacio entre barras para que se vea más estético
)

fig1.show()

In [18]:
# Clasificación del volumen de vehículos disponibles según su tipo de carrocería.

type_counts = df_vehicles['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']

fig2 = px.bar(type_counts, x='type', y='count',
title='Volumen de inventario por tipo de vehículo',
labels={'type': 'Tipo de Vehículo', 'count': 'Cantidad'},
color='count',
color_continuous_scale='Reds')
fig2.show()

In [19]:
# Condición vs. precio

fig3 = px.violin(df_vehicles, x='condition', y='price', color='condition',
box=True,
title='Distribución de precios según condición del vehículo',
labels={'condition': 'Estado', 'price': 'Precio ($)'},
points="all")
fig3.show()

In [20]:
# Relaciones técnicas: precio, cilindros y tracción

fig4 = px.scatter(df_vehicles, x='cylinders', y='price',
color='is_4wd',
size='days_listed',
hover_data=['model'],
title='Precio vs cilindros (Color: 4x4 | Tamaño: Días en Lista)',
labels={'cylinders': 'Número de Cilindros', 'price': 'Precio ($)', 'is_4wd': 'Tracción 4x4'})
fig4.show()